# 02 · Gemma — iteración manual del pipeline de texto

En el benchmark inicial se compararon tres familias de modelos abiertas sobre los mismos chunks del corpus: **Qwen3-4B-Instruct-2507, Gemma 3 4B IT y Mistral 7B Instruct v0.3**.

Para continuar el prototipado se selecciona **Gemma 3 4B IT** porque, en la muestra inicial:

- produjo salida JSON válida de forma consistente;
- mostró buena adherencia al texto fuente;
- tuvo una latencia competitiva;
- con 4B parámetros resulta razonable para iterar en Google Colab.

Esta decisión es **pragmática y provisional**: el objetivo de este notebook no es establecer un benchmark exhaustivo, sino depurar manualmente el comportamiento del pipeline antes de automatizarlo.

## Dos estrategias que vamos a comparar

### Pipeline 1 — Extract → Select → Write
`chunk → hasta 3 ideas → selección de la idea relevante → texto final corto`

Esta estrategia fuerza al modelo a separar extracción factual y redacción.

### Pipeline 2 — Direct summary
`chunk completo → texto final de ~30 segundos`

Esta estrategia sirve como baseline simple para comprobar si la descomposición del Pipeline 1 aporta valor.

In [1]:
!pip -q install -U "transformers>=4.51.0" accelerate bitsandbytes huggingface_hub pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 31.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
dask-cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
cudf-cu13 26.6.0 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.


## 1. Clonar FinTok y cargar el corpus

In [2]:
from pathlib import Path
import json, random, gc, re, time
import numpy as np
import pandas as pd
import torch

# Clonar solo si todavía no existe en esta sesión de Colab
if not Path("/content/FinTok").exists():
    !git clone https://github.com/mirdbg/FinTok.git

REPO_DIR = Path("/content/FinTok")
ZIP_PATH = REPO_DIR / "data" / "corpus_miax_2026.zip"
WORK_DIR = Path("/content/corpus_miax_2026")

WORK_DIR.mkdir(parents=True, exist_ok=True)

assert ZIP_PATH.exists(), f"No encuentro: {ZIP_PATH}"

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Corpus:", ZIP_PATH)

Cloning into 'FinTok'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 4), reused 18 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 1.76 MiB | 23.65 MiB/s, done.
Resolving deltas: 100% (4/4), done.
GPU: Tesla T4
Corpus: /content/FinTok/data/corpus_miax_2026.zip


In [3]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extractall(WORK_DIR)

with open(WORK_DIR / "chunks.jsonl", "r", encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]

df = pd.DataFrame(chunks)

print(f"Chunks cargados: {len(df):,}")
display(df[["chunk_id", "ticker", "fiscal_year", "item", "n_tokens", "contiene_tabla"]].head())

Chunks cargados: 1,749


,chunk_id,ticker,fiscal_year,item,n_tokens,contiene_tabla
0,NVDA-2024-1A-0000,NVDA,2024,1A,475,False
1,NVDA-2024-1A-0001,NVDA,2024,1A,278,False
2,NVDA-2024-1A-0002,NVDA,2024,1A,500,False
3,NVDA-2024-1A-0003,NVDA,2024,1A,500,False
4,NVDA-2024-1A-0004,NVDA,2024,1A,99,False


## 2. Cargar Gemma

El notebook usa cuantización 4-bit para reducir VRAM.

`HF_TOKEN` debe estar guardado en **Secrets** de Colab y tener activado *Acceso desde el cuaderno*.

In [4]:
from huggingface_hub import login
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
print("Hugging Face autenticado.")

Hugging Face autenticado.


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "google/gemma-3-4b-it"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype="auto",
    quantization_config=quant_config,
)

model.eval()
print("Modelo cargado:", MODEL_ID)

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.16M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 33.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Modelo cargado: google/gemma-3-4b-it


## 3. Función común de generación

In [6]:
def ask_gemma(system_prompt, user_prompt, max_new_tokens=600):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    start = time.perf_counter()

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    elapsed = time.perf_counter() - start
    generated = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(generated, skip_special_tokens=True).strip(), elapsed

## 4. Elegir manualmente un chunk

Durante esta fase queremos **iterar manualmente**, así que no hacemos todavía un bucle por los 1.749 chunks.

Puedes filtrar el DataFrame, leer candidatos y cambiar `CHUNK_ID` tantas veces como quieras.

In [7]:
# Algunos candidatos para explorar
display(
    df.sample(15, random_state=42)[
        ["chunk_id", "ticker", "fiscal_year", "item", "n_tokens", "contiene_tabla"]
    ]
)

,chunk_id,ticker,fiscal_year,item,n_tokens,contiene_tabla
660,AAPL-2024-8-0043,AAPL,2024,8,278,False
1353,META-2025-7-0010,META,2025,7,500,False
861,GOOGL-2024-8-0037,GOOGL,2024,8,370,True
111,NVDA-2024-8-0035,NVDA,2024,8,453,True
332,MSFT-2024-7-0001,MSFT,2024,7,500,False
1005,GOOGL-2025-8-0026,GOOGL,2025,8,500,True
1165,META-2024-7-0020,META,2024,7,157,True
109,NVDA-2024-8-0033,NVDA,2024,8,484,True
135,NVDA-2024-8-0059,NVDA,2024,8,470,False
1440,META-2025-8-0058,META,2025,8,500,False


In [8]:
# CAMBIA ESTE ID para probar otro chunk
CHUNK_ID = df.sample(1, random_state=42).iloc[0]["chunk_id"]

row = df.loc[df["chunk_id"] == CHUNK_ID].iloc[0]
chunk_text = row["texto"]

print("CHUNK:", row["chunk_id"])
print("TICKER:", row["ticker"])
print("AÑO:", row["fiscal_year"])
print("ITEM:", row["item"])
print("=" * 100)
print(chunk_text)

CHUNK: AAPL-2024-8-0043
TICKER: AAPL
AÑO: 2024
ITEM: 8
Definition and Limitations of Internal Control Over Financial Reporting

A company’s internal control over financial reporting is a process designed to provide reasonable assurance regarding the reliability of financial reporting and the preparation of financial statements for external purposes in accordance with generally accepted accounting principles. A company’s internal control over financial reporting includes those policies and procedures that (1) pertain to the maintenance of records that, in reasonable detail, accurately and fairly reflect the transactions and dispositions of the assets of the company; (2) provide reasonable assurance that transactions are recorded as necessary to permit preparation of financial statements in accordance with generally accepted accounting principles, and that receipts and expenditures of the company are being made only in accordance with authorizations of management and directors of the com

# PIPELINE 1 · Extract → Select → Write

Esta celda realiza deliberadamente **tres operaciones dentro de una única llamada**:

1. Extrae hasta tres ideas factuales de 3 chunks consecutivos.
2. Decide si alguna tiene suficiente valor para FinTok.
3. Si existe una idea adecuada, redacta un texto corto y autosuficiente.

El modelo puede devolver `DISCARD`. No queremos obligarlo a fabricar contenido a partir de un fragmento irrelevante.

Durante la iteración manual pedimos también que muestre las ideas candidatas y la razón de selección. Más adelante podremos ocultar esa traza y conservar únicamente `final_text`.

In [11]:
display(
    df[
        ["chunk_id", "ticker", "fiscal_year", "item", "posicion", "n_tokens"]
    ].sample(5, random_state=42)
)

,chunk_id,ticker,fiscal_year,item,posicion,n_tokens
660,AAPL-2024-8-0043,AAPL,2024,8,43,278
1353,META-2025-7-0010,META,2025,7,10,500
861,GOOGL-2024-8-0037,GOOGL,2024,8,37,370
111,NVDA-2024-8-0035,NVDA,2024,8,35,453
332,MSFT-2024-7-0001,MSFT,2024,7,1,500


In [12]:
# Elegimos manualmente el chunk CENTRAL y añadimos los adyacentes

CHUNK_ID = "AAPL-2025-7-0009"  # cambia aquí el chunk que quieras probar

target = df.loc[df["chunk_id"] == CHUNK_ID].iloc[0]

# Nos quedamos en la misma empresa, año e Item
section_df = (
    df[
        (df["ticker"] == target["ticker"]) &
        (df["fiscal_year"] == target["fiscal_year"]) &
        (df["item"] == target["item"])
    ]
    .sort_values("posicion")
    .reset_index(drop=True)
)

# Localizamos el chunk elegido
center_idx = section_df.index[
    section_df["chunk_id"] == CHUNK_ID
][0]

# Chunk anterior + central + posterior
start = max(0, center_idx - 1)
end = min(len(section_df), center_idx + 2)

context_df = section_df.iloc[start:end]

display(
    context_df[
        ["chunk_id", "ticker", "fiscal_year", "item", "posicion"]
    ]
)

context_text = "\n\n".join(
    f"""--- CHUNK {row['chunk_id']} ---
{row['texto']}"""
    for _, row in context_df.iterrows()
)

print(context_text)

,chunk_id,ticker,fiscal_year,item,posicion
8,AAPL-2025-7-0008,AAPL,2025,7,8
9,AAPL-2025-7-0009,AAPL,2025,7,9
10,AAPL-2025-7-0010,AAPL,2025,7,10


--- CHUNK AAPL-2025-7-0008 ---
Capital Return Program

In addition to its contractual cash requirements, the Company has an authorized share repurchase program. The program does not obligate the Company to acquire a minimum amount of shares. As of September 27, 2025, the Company’s quarterly cash dividend was $0.26 per share. The Company intends to increase its dividend on an annual basis, subject to declaration by the Board.

In May 2025, the Company announced a new share repurchase program of up to $100 billion and raised its quarterly dividend from $0.25 to $0.26 per share beginning in May 2025. During 2025, the Company repurchased $89.3 billion of its common stock and paid dividends and dividend equivalents of $15.4 billion.

Recent Accounting Pronouncements

Internal-Use Software

In September 2025, the Financial Accounting Standards Board (“FASB”) issued Accounting Standards Update (“ASU”) No. 2025-06, Intangibles—Goodwill and Other—Internal-Use Software (Subtopic 350-40): Targete

### PASO 1: EXTRAER

In [13]:
EXTRACT_SYSTEM = """
You are a financial analyst specialized in SEC 10-K filings.

Read the supplied excerpts from three consecutive chunks of the same 10-K section.

Extract up to 3 distinct, self-contained factual ideas that are explicitly
supported by the text.

Prioritize:
- important quantitative results or changes,
- year-over-year developments,
- business drivers,
- strategic developments,
- customer or supplier dependencies,
- material risks,
- unusual or potentially meaningful facts.

Rules:
- Use ONLY information contained in the supplied excerpts.
- Do not use external knowledge.
- Do not speculate.
- Do not exaggerate.
- Combine information across the three chunks when they clearly belong to
  the same idea.
- Do not create separate ideas just because the same information appears
  across overlapping chunks.
- Return fewer than 3 ideas if necessary.
- If nothing meaningful exists, return an empty list.
- This is factual extraction, NOT social-media copywriting.

Return ONLY valid JSON:

{
  "ideas": [
    {
      "idea": "...",
      "evidence": "...",
      "source_chunks": ["chunk_id"]
    }
  ]
}
""".strip()


extract_user = f"""
COMPANY/TICKER: {target['ticker']}
FISCAL YEAR: {target['fiscal_year']}
10-K ITEM: {target['item']}

CONSECUTIVE 10-K EXCERPTS:
----------------
{context_text}
----------------
""".strip()


extract_output, extract_latency = ask_gemma(
    EXTRACT_SYSTEM,
    extract_user,
    max_new_tokens=550
)

print(f"Latencia: {extract_latency:.2f}s")
print(extract_output)

Latencia: 65.25s
```json
{
  "ideas": [
    {
      "idea": "Apple repurchased $89.3 billion of its common stock during 2025 and paid $15.4 billion in dividends and dividend equivalents.",
      "evidence": "CHUNK AAPL-2025-7-0008: ‘During 2025, the Company repurchased $89.3 billion of its common stock and paid dividends and dividend equivalents of $15.4 billion.’",
      "source_chunks": ["AAPL-2025-7-0008"]
    },
    {
      "idea": "The company’s quarterly dividend increased from $0.25 to $0.26 per share in May 2025.",
      "evidence": "CHUNK AAPL-2025-7-0008: ‘In May 2025, the Company announced a new share repurchase program of up to $100 billion and raised its quarterly dividend from $0.25 to $0.26 per share beginning in May 2025.’",
      "source_chunks": ["AAPL-2025-7-0008"]
    },
    {
      "idea": "The Company will adopt ASU 2024-03 in its fourth quarter of 2028 using a prospective transition method, requiring disaggregation of income statement expenses.",
      "evidence"

### PASO 2: SELECCIONAR

In [14]:
SELECT_SYSTEM = """
You are the financial editor of FinTok.

Your task is to decide whether any of the candidate ideas extracted from a
10-K is genuinely worth communicating to a general but financially interested
audience.

A fact being correct does NOT automatically make it interesting.

Good candidates tend to involve:
- substantial changes in the business,
- striking financial figures,
- important growth or declines,
- meaningful risks,
- important dependencies,
- strategic changes,
- information that reveals something useful about how the company operates.

Reject ideas that are mainly:
- administrative,
- routine accounting language,
- legal boilerplate,
- too technical without broader relevance,
- impossible to understand without extensive context,
- repetitive or trivial.

Be selective.

Select AT MOST ONE idea.

Use the original excerpts to verify that the selected idea is properly
supported.

Return ONLY valid JSON:

{
  "decision": "KEEP or DISCARD",
  "selected_idea": "..." or null,
  "reason": "..."
}
""".strip()


select_user = f"""
ORIGINAL 10-K EXCERPTS:
----------------
{context_text}
----------------

CANDIDATE IDEAS:
----------------
{extract_output}
----------------
""".strip()


select_output, select_latency = ask_gemma(
    SELECT_SYSTEM,
    select_user,
    max_new_tokens=300
)

print(f"Latencia: {select_latency:.2f}s")
print(select_output)

Latencia: 17.44s
```json
{
  "decision": "KEEP",
  "selected_idea": "Apple repurchased $89.3 billion of its common stock during 2025 and paid $15.4 billion in dividends and dividend equivalents.",
  "reason": "This is a significant financial figure that demonstrates the company's financial health and commitment to returning value to shareholders. It’s a clear and impactful statement of the company’s actions, easily understandable by a general audience interested in finance. The other ideas are important, but this one is the most immediately striking and easily communicated."
}
```


### PASO 3: ESCRIBIR

In [18]:
WRITE_SYSTEM = """
You are a financial content writer for FinTok, a platform that transforms
official 10-K filings into short, engaging and easy-to-understand financial content.

Turn the selected financial idea into a natural spoken script for a short video.

Use the original 10-K excerpts as the ONLY factual source.

The script must follow this structure:

1. HOOK
Start immediately with ONE short and engaging question or surprising factual
statement based on the selected idea.

The hook should make the viewer curious and want to keep watching.
You may use formats such as:
- "Did you know that...?"
- "What if I told you that...?"
- "How much...?"
- "Why did...?"
- "One number in [company]'s latest filing stands out..."

Do NOT use misleading clickbait and do NOT exaggerate beyond what is supported
by the source.

2. FINTOK INTRO
After the hook, include EXACTLY:

"This is FinTok — your daily dose of financial knowledge."

3. EXPLANATION
Explain the selected idea clearly and naturally.

The explanation must:
- focus on ONE main financial story;
- identify the company;
- preserve the most relevant figures and context;
- explain the fact in language understandable to a general interested audience;
- sound natural when spoken aloud;
- avoid unnecessary financial jargon;
- avoid simply copying regulatory language;
- avoid investment advice;
- avoid unsupported interpretations;
- use ONLY information supported by the supplied 10-K excerpts.

Do not try to summarize everything in the excerpts.
Only include information that helps explain the selected story.

4. FINTOK OUTRO
End EXACTLY with:

"That's your FinTok for today. See you in the next one."

The complete script should take approximately 30 seconds to read aloud.
Aim for roughly 90-120 words in total.

Return ONLY the final spoken script.
Do not include headings, labels, explanations or quotation marks.
""".strip()


write_user = f"""
SELECTED IDEA:
----------------
{select_output}
----------------

SOURCE 10-K EXCERPTS:
----------------
{context_text}
----------------
""".strip()


write_output, write_latency = ask_gemma(
    WRITE_SYSTEM,
    write_user,
    max_new_tokens=220
)

print(f"Latencia: {write_latency:.2f}s")
print()
print(write_output)
print()
print("Palabras:", len(write_output.split()))

Latencia: 19.62s

Did you know that Apple spent a massive $89.3 billion in 2025 just buying back its own stock and paying dividends?

This is FinTok — your daily dose of financial knowledge.

During 2025, Apple repurchased $89.3 billion of its common stock and paid dividends and dividend equivalents of $15.4 billion, as detailed in their 2025 filing. This significant amount demonstrates the company’s financial health and its commitment to returning value to shareholders. They also raised their quarterly dividend from $0.25 to $0.26 per share.

That's your FinTok for today. See you in the next one.

Palabras: 95


## Próximos pasos

Cuando el comportamiento sea satisfactorio, este notebook se puede convertir en un pipeline automático:

`todos los chunks → Pipeline 1 → KEEP → generación del guion definitivo → voz / imagen / vídeo`

Antes de automatizarlo conviene decidir manualmente:
1. qué entendemos por una idea suficientemente interesante;
2. cuánto debe durar el contenido;
3. qué tono editorial queremos;
4. si una sola idea por vídeo es la estrategia adecuada.